# Quadratic XX Chain With Local Loss

This notebook shows when the quadratic Lindblad interface is worth using. We consider a four-site XX chain with a single fermion-loss channel at site 1, and compare two calculations:

1. full many-body density-matrix evolution with `lindblad`,
2. covariance-matrix evolution with `quadraticlindblad`.

The model is small enough that we can compare both directly, but structured enough that the quadratic route already looks much cleaner.

## Physical picture

The XX chain maps to free fermions under Jordan-Wigner. A local fermion loss channel stays quadratic in that language, so the covariance matrix contains all the information needed for one-body observables such as site occupations.

That means we can benchmark the compact quadratic description against the full many-body evolution and verify that they agree.

In [ ]:
import Pkg

function find_repo_root(start = pwd())
    dir = abspath(start)
    while true
        if isfile(joinpath(dir, "Project.toml")) && isfile(joinpath(dir, "src", "EDKit.jl"))
            return dir
        end
        parent = dirname(dir)
        parent == dir && error("Could not locate EDKit.jl repo root from $(start)")
        dir = parent
    end
end

const DEV = true
const REPO_ROOT = find_repo_root()
Pkg.activate(REPO_ROOT)
if DEV
    pushfirst!(LOAD_PATH, REPO_ROOT)
end

using EDKit
using LinearAlgebra
using Statistics


In [ ]:
function quadratic_chain(L, r, a, b)
    H1 = zeros(2L, 2L)
    for i in 1:L-1
        H1[i, i + 1 + L] = 2r[i]
        H1[i + 1 + L, i] = -2r[i]
        H1[i + 1, i + L] = 2r[i]
        H1[i + L, i + 1] = -2r[i]
    end

    L1 = zeros(ComplexF64, 2L, L)
    for i in 1:L
        L1[i, i] = a[i]
        L1[i + L, i] = b[i]
    end

    ql = quadraticlindblad(H1, L1)
    cm0 = covariancematrix([mod(i, 2) for i in 1:L])
    ql, cm0
end

function manybody_chain(L, r, a, b)
    mats = [spin((r[i], "XX"), (r[i], "YY"), D = 2) for i in 1:L-1]
    inds = [[i, i + 1] for i in 1:L-1]
    H = Array(operator(mats, inds, L))

    jumps = Matrix{ComplexF64}[]
    for i in 1:L
        string_z = if i == 1
            Matrix{Float64}(I, 1, 1)
        elseif i == 2
            [-1.0 0.0; 0.0 1.0]
        else
            reduce(kron, fill([-1.0 0.0; 0.0 1.0], i - 1))
        end
        local_jump = spin((a[i], "X"), (b[i], "Y"), D = 2)
        right_eye = Matrix{Float64}(I, 2^(L - i), 2^(L - i))
        push!(jumps, kron(string_z, local_jump, right_eye))
    end

    lb = lindblad(H, jumps)
    dm0 = densitymatrix(index([mod(i - 1, 2) for i in 1:L]), L)
    lb, dm0
end

function site_densities(dm, L)
    vals = zeros(L)
    for i in 1:L
        n_op = Hermitian(Array(operator([1.0 0.0; 0.0 0.0], [i], L)))
        vals[i] = expectation(n_op, dm)
    end
    vals
end


## Parameter choice

We use a uniform XX chain and put a single loss channel on the first site. In fermionic language, that is the cleanest local dissipative process. The coefficients `a[1] = √γ / 2` and `b[1] = i √γ / 2` implement the annihilation operator after the Jordan-Wigner mapping used by the package.

In [ ]:
L = 4
J = 0.6
gamma = 0.3
dt = 0.02
steps = 80

r = fill(J, L - 1)
a = zeros(ComplexF64, L)
b = zeros(ComplexF64, L)
a[1] = sqrt(gamma) / 2
b[1] = 1im * sqrt(gamma) / 2

ql, cm = quadratic_chain(L, r, a, b)
lb, dm = manybody_chain(L, r, a, b)


In [ ]:
function evolve_both(ql, cm0, lb, dm0; dt, steps, order = 8)
    times = collect(0:steps) .* dt
    quadratic_profiles = zeros(length(times), cm0.N)
    manybody_profiles = zeros(length(times), cm0.N)

    cm = cm0
    dm = dm0
    quadratic_profiles[1, :] = real.(diag(fermioncorrelation(cm, 1)))
    manybody_profiles[1, :] = site_densities(dm, cm0.N)

    for step in 1:steps
        cm = ql(cm, dt, order = order)
        dm = lb(dm, dt, order = order)
        quadratic_profiles[step + 1, :] = real.(diag(fermioncorrelation(cm, 1)))
        manybody_profiles[step + 1, :] = site_densities(dm, cm0.N)
    end

    (; times, quadratic_profiles, manybody_profiles, final_density = Array(dm))
end

results = evolve_both(ql, cm, lb, dm; dt = dt, steps = steps)
errors = [norm(results.quadratic_profiles[i, :] - results.manybody_profiles[i, :]) for i in eachindex(results.times)]


## Agreement check

The occupation profiles should agree essentially pointwise. Any residual difference here is numerical integration error, not a physical discrepancy between the two formalisms.

In [ ]:
summary = (
    final_quadratic = vec(results.quadratic_profiles[end, :]),
    final_manybody = vec(results.manybody_profiles[end, :]),
    max_profile_error = maximum(errors),
    final_profile_error = errors[end],
)
summary


In [ ]:
try
    using Plots
    p1 = plot(results.times, results.quadratic_profiles[:, 1]; label = "quadratic n1", xlabel = "time", ylabel = "occupation")
    plot!(p1, results.times, results.manybody_profiles[:, 1]; label = "many-body n1", linestyle = :dash)
    plot!(p1, results.times, results.quadratic_profiles[:, 2]; label = "quadratic n2")
    plot!(p1, results.times, results.manybody_profiles[:, 2]; label = "many-body n2", linestyle = :dash)

    p2 = plot(1:L, results.quadratic_profiles[end, :]; label = "quadratic final", marker = :circle, xlabel = "site", ylabel = "occupation")
    plot!(p2, 1:L, results.manybody_profiles[end, :]; label = "many-body final", marker = :diamond)

    p3 = plot(results.times, errors; label = "profile error", xlabel = "time", ylabel = "||n_quad - n_mb||")
    plot(p1, p2, p3; layout = (3, 1), size = (850, 900))
catch err
    @info "Plots not available; returning arrays instead" error = err
    (
        quadratic_profiles = results.quadratic_profiles,
        manybody_profiles = results.manybody_profiles,
        errors = errors,
    )
end


## Interpretation

This benchmark is the practical reason to keep the quadratic interface around. The many-body evolution is the direct and general route, but once the model is genuinely quadratic, `quadraticlindblad` captures the same one-body physics in a much more structured form.

That makes it a better starting point for free-fermion transport, relaxation, and steady-state calculations, while the full `lindblad` interface remains the general-purpose fallback.